In [0]:
from pyspark.sql import functions as F

REF = "/Volumes/voebem/bronze/arquivos/referencias"

SEM_ASPAS = chr(0)

In [0]:
aerodromos = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", "true")
    .option("skipRows", 1)
    .option("encoding", "ISO-8859-1") 
    .option("quote", SEM_ASPAS)
    .load(f"{REF}/AerodromosPublicos.csv")
)

aerodromos = aerodromos.select(
    F.col("`Código OACI`").alias("icao"),
    F.col("CIAD").alias("ciad"),
    F.col("Nome").alias("nome"),
    F.col("`Município`").alias("municipio"),
    F.col("UF").alias("uf"),
    F.col("`Município Servido`").alias("municipio_servido"),
    F.col("`UF Servido`").alias("uf_servido"),
    F.col("Latitude").alias("latitude"),
    F.col("Longitude").alias("longitude"),
    F.col("Altitude").alias("altitude"),
    F.col("`Situação`").alias("situacao"),
).withColumn("_ingerido_em", F.current_timestamp())

aerodromos.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("voebem.bronze.aerodromos")

print(f"bronze.aerodromos: {spark.table('voebem.bronze.aerodromos').count():,} linhas")
display(spark.sql("SELECT icao, nome, municipio, uf FROM voebem.bronze.aerodromos WHERE icao IN ('SBRB','SBGR','SBSP','SBFZ')"))


bronze.aerodromos: 496 linhas


icao,nome,municipio,uf
SBRB,Plácido de Castro,RIO BRANCO,Acre
SBFZ,Pinto Martins,FORTALEZA,Ceará
SBSP,São Paulo/Congonhas - Deputado Freitas Nobre,SÃO PAULO,São Paulo
SBGR,Guarulhos - Governador André Franco Montoro,GUARULHOS,São Paulo


In [0]:
def ler_empresas(arquivo: str):
    """Le um cadastro de empresas. Sem uniao, sem enriquecimento: uma tabela por arquivo."""
    return (
        spark.read.format("csv")
        .option("sep", ";")
        .option("header", "true")
        .option("skipRows", 1)
        .option("encoding", "UTF-8")
        .option("quote", '"')
        .load(f"{REF}/{arquivo}")
        .select(
            F.col("ICAO").alias("icao"),
            F.col("Estrangeira").alias("sigla_iata"),
            F.col("Razao").alias("razao_social"),
            F.col("Servico").alias("servico"),
            F.col("Cidade").alias("cidade"),
            F.col("UF").alias("uf"),
            F.col("Ativa").alias("situacao"),
        )
        .withColumn("_arquivo_origem", F.lit(arquivo))
        .withColumn("_ingerido_em", F.current_timestamp())
    )


for arquivo, tabela in [
    ("pda_empresas_aereas_nacionais.csv",    "voebem.bronze.empresas_nacionais"),
    ("pda_empresas_aereas_estrangeiros.csv", "voebem.bronze.empresas_estrangeiras"),
]:
    ler_empresas(arquivo).write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(tabela)
    print(f"{tabela}: {spark.table(tabela).count():,} linhas")

voebem.bronze.empresas_nacionais: 729 linhas
voebem.bronze.empresas_estrangeiras: 148 linhas


In [0]:
display(spark.sql("""
    SELECT 'empresas_nacionais' AS tabela, COUNT(*) AS linhas,
           COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END) AS com_icao
    FROM voebem.bronze.empresas_nacionais
    UNION ALL
    SELECT 'empresas_estrangeiras', COUNT(*),
           COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END)
    FROM voebem.bronze.empresas_estrangeiras
"""))

tabela,linhas,com_icao
empresas_nacionais,729,20
empresas_estrangeiras,148,147


In [0]:
display(spark.sql("""
    SELECT icao, razao_social, servico, uf, situacao
    FROM voebem.bronze.empresas_nacionais
    WHERE icao IN ('GLO','TAM','AZU','PAM')
    ORDER BY icao
"""))

display(spark.sql("""
    SELECT icao, razao_social, servico, situacao
    FROM voebem.bronze.empresas_estrangeiras
    WHERE icao IN ('AAL','TAP','AVA','ARG')
    ORDER BY icao
"""))

icao,razao_social,servico,uf,situacao
AZU,AZUL LINHAS AÉREAS BRASILEIRAS S/A,"TRANSPORTE AÉREO NÃO REGULAR, TRANSPORTE AÉREO REGULAR",SP,ATIVA
GLO,GOL LINHAS AÉREAS S.A. (EX- VRG LINHAS AÉREAS S.A.),"TRANSPORTE AÉREO NÃO REGULAR, TRANSPORTE AÉREO REGULAR",RJ,ATIVA
TAM,TAM LINHAS AÉREAS S.A.,TRANSPORTE AÉREO REGULAR,SP,ATIVA


icao,razao_social,servico,situacao
AAL,"AMERICAN AIRLINES, INC.",ESTRANGEIRA REGULAR,ATIVA
ARG,AEROLINEAS ARGENTINAS S/A,ESTRANGEIRA REGULAR,ATIVA
AVA,AEROVIAS DEL CONTINENTE AMERICANO S.A. AVIANCA,ESTRANGEIRA REGULAR,ATIVA
TAP,TAP - TRANSPORTES AÉREOS PORTUGUESES S/A,ESTRANGEIRA REGULAR,ATIVA


In [0]:
CODIGOS = [
    ("codigo_di", "0", "Etapa Regular"),
    ("codigo_di", "2", "Etapa Extra"),
    ("codigo_di", "3", "Etapa de Retorno"),
    ("codigo_di", "4", "Inclusão de Etapa"),
    ("codigo_di", "6", "Etapa Não Remunerada Sem Transporte de Objetos"),
    ("codigo_di", "7", "Etapa de Voo de Fretamento"),
    ("codigo_di", "9", "Etapa de Voo Charter"),
    ("codigo_di", "D", "Etapa de Voo Duplicada"),
    ("codigo_di", "E", "Etapa Não Remunerada Com Transporte de Objetos"),
    ("codigo_tipo_linha", "N", "Doméstica Mista"),
    ("codigo_tipo_linha", "C", "Doméstica Cargueira"),
    ("codigo_tipo_linha", "I", "Internacional Mista"),
    ("codigo_tipo_linha", "G", "Internacional Cargueira"),
]

codigos = spark.createDataFrame(CODIGOS, "dominio string, codigo string, descricao string")
codigos.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("voebem.bronze.codigos_operacao")

print(f"bronze.codigos_operacao: {spark.table('voebem.bronze.codigos_operacao').count()} linhas")
display(spark.table("voebem.bronze.codigos_operacao"))


bronze.codigos_operacao: 13 linhas


dominio,codigo,descricao
codigo_di,0,Etapa Regular
codigo_di,2,Etapa Extra
codigo_di,3,Etapa de Retorno
codigo_di,4,Inclusão de Etapa
codigo_di,6,Etapa Não Remunerada Sem Transporte de Objetos
codigo_di,7,Etapa de Voo de Fretamento
codigo_di,9,Etapa de Voo Charter
codigo_di,D,Etapa de Voo Duplicada
codigo_di,E,Etapa Não Remunerada Com Transporte de Objetos
codigo_tipo_linha,N,Doméstica Mista


In [0]:
display(spark.sql("SHOW TABLES IN voebem.bronze"))

database,tableName,isTemporary
bronze,aerodromos,false
bronze,codigos_operacao,false
bronze,empresas_estrangeiras,false
bronze,empresas_nacionais,false
bronze,vra,false


In [0]:
display(spark.sql("""
    SELECT version, timestamp, operation,
           operationMetrics.numOutputRows AS linhas_escritas
    FROM (DESCRIBE HISTORY voebem.bronze.vra)
    ORDER BY version
"""))

version,timestamp,operation,linhas_escritas
0,2026-09-17T16:56:30.000Z,CREATE OR REPLACE TABLE AS SELECT,1014705
1,2026-09-17T16:57:53.000Z,SET TBLPROPERTIES,null
2,2026-09-17T16:58:01.000Z,CREATE OR REPLACE TABLE AS SELECT,1014705
3,2026-09-17T16:58:04.000Z,SET TBLPROPERTIES,null
4,2026-09-17T16:58:50.000Z,SET TBLPROPERTIES,null


In [0]:
display(spark.sql("""
    SELECT 'versao 0 (1a carga)'   AS versao,
           COUNT(*)                AS linhas,
           MIN(_ingerido_em)       AS ingerido_em
    FROM voebem.bronze.vra VERSION AS OF 0
    UNION ALL
    SELECT 'versao atual', COUNT(*), MIN(_ingerido_em)
    FROM voebem.bronze.vra
"""))

versao,linhas,ingerido_em
versao 0 (1a carga),1014705,2026-09-17T16:56:23.300Z
versao atual,1014705,2026-09-17T16:57:57.913Z


In [0]:
for tabela, comentario in [
    ("voebem.bronze.aerodromos",
     "Bronze - cadastro de aerodromos publicos da ANAC, como chegou. Chave: codigo ICAO (OACI). "
     "Cobre apenas aerodromos brasileiros - aeroportos estrangeiros do VRA nao estao aqui."),
    ("voebem.bronze.empresas_nacionais",
     "Bronze - cadastro de empresas aereas NACIONAIS da ANAC, como chegou. Chave: codigo ICAO. "
     "Nao unir com empresas_estrangeiras nesta camada: a uniao e feita na silver."),
    ("voebem.bronze.empresas_estrangeiras",
     "Bronze - cadastro de empresas aereas ESTRANGEIRAS autorizadas a operar no Brasil, como chegou. "
     "Chave: codigo ICAO. Cadastro separado do nacional na origem, mantido separado no bronze."),
    ("voebem.bronze.codigos_operacao",
     "Bronze - seed table curada a partir da pagina de descricao de variaveis da ANAC. "
     "Traduz codigo_di e codigo_tipo_linha para descricao em portugues."),
]:
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")

print("comentarios aplicados")

comentarios aplicados
